In [4]:
import struct
import numpy as np


"""Попытка прочитать базовые поля из сериализованного DefectResults"""
with open('./thicks_export/thick_378451.raw', 'rb') as f:
    # Пропуск заголовка сериализации (~20-50 байт)
    # Это упрощённый пример — реальная структура сложнее
    try:
        x = struct.unpack('>i', f.read(4))[0]      # big-endian int
        y = struct.unpack('>i', f.read(4))[0]
        thicknom = struct.unpack('>d', f.read(8))[0]  
       
        print(f"Размеры: {x}×{y} толщина {thicknom}")
        #return doubles
    except Exception as e:
        print(f"⚠️ Не удалось распарсить: {e}")
#return None

Размеры: 188×8 толщина 5.5


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import struct
import os

def read_raw_file(filepath):
    """Чтение бинарного файла, созданного Java DataOutputStream (новый формат)"""
    with open(filepath, 'rb') as f:
        # Чтение заголовка (Java использует big-endian)
        x = struct.unpack('>i', f.read(4))[0]           # counts.length (X)
        y = struct.unpack('>i', f.read(4))[0]           # counts[0].length (Y)
        thicknom = struct.unpack('>d', f.read(8))[0] 
        
        print(f"📊 Заголовок: {x}×{y}")
        
        # Чтение данных: double (8 байт, big-endian)
        # Данные представляют собой усреднённые значения [X][Y]
        total_elements = x * y
        data_flat = np.frombuffer(f.read(8 * total_elements), dtype='>f8')
        
        # Формирование 2D массива [X][Y]
        data = data_flat.reshape((x, y))
        
        return {
            'x': x, 'y': y,'thicknom':thicknom,
            'data': data
        }


def create_defect_cmap(threshold=1.0):
    """Создание цветовой карты: 
    - 0 → белый
    - значения < порога → градиент красного (от белого к красному)
    - значения > порога → градиент зелёного (от светло-зелёного к насыщенному)
    """
    colors = []
    
    # Белая точка для нуля (начало)
    colors.append((1.0, 1.0, 1.0))  # белый
    
    # Вычисляем пропорции для красного и зелёного градиентов
    # threshold определяет точку перехода от красного к зелёному
    n_red = int(256 * threshold)  # количество оттенков красного до порога
    n_green = int(256 * (1-threshold)) # количество оттенков зелёного после порога
    
    # Градиент красного для значений от 0 до threshold (исключая 0)
    # От почти белого к насыщенному красному
    for i in range(max(1, n_red)):
        intensity = 0.95 - 0.95 * (i / max(1, n_red - 1))  # от почти белого к красному
        colors.append((1.0, intensity, intensity))  # (R, G, B) - розовый/красный градиент
    
    # Насыщенный красный на пороге
    colors.append((1.0, 0.0, 0.0))  # чистый красный на threshold
    
    # Градиент зелёного для значений > threshold
    # От светло-зелёного к насыщенному зелёному
    for i in range(n_green):
        intensity = 0.5 + 0.5 * (i / max(1, n_green - 1))  # от 0.5 к 1.0
        colors.append((0, intensity, 0))  # (R, G, B)
    
    return mcolors.ListedColormap(colors)


def visualize_heatmap(data_2d, output_path=None, threshold=1.0, 
                      x_size=None, y_size=None, title="Толщинометрия"):
    """
    Визуализация 2D карты значений.
    data_2d: массив [X][Y] с усреднёнными значениями
    """
    x, y = data_2d.shape
    
    
    
    # Нормализация: 0 → белый, < threshold → красный градиент, > threshold → зелёный градиент
    vmin = min(0, np.min(data_2d))
    vmax = max(threshold * 1.5, np.max(data_2d))
   # vmax = np.max(data_2d)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    
    # Создаём цветовую карту
    cmap = create_defect_cmap(threshold/vmax)
    
    # Построение изображения
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # transpose для корректной ориентации (Y по вертикали)
    # origin='lower' чтобы (0,0) было в левом нижнем углу
    im = ax.imshow(data_2d.T, cmap=cmap, norm=norm,
                   origin='lower', interpolation='bilinear',
                   extent=[0, x_size or x, 0, y_size or y*5])
    
    # Добавляем линию порога на цветовой шкале
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
   # cbar.set_label('Нормированное значение', rotation=270, labelpad=20)
    
    # Добавляем маркер порога на colorbar
    cbar.ax.axhline(y= threshold, #(threshold - vmin) / (vmax - vmin), 
                    color='black', linestyle='--', linewidth=1)
    cbar.ax.text(2.02, threshold, #(threshold - vmin) / (vmax - vmin), 
                 f'Порог: {threshold}', va='center', fontsize=9)
    
    plt.title(f'{title}\n(цвет: 🟢 норма > {threshold} > 🔴 дефект)')
    plt.xlabel('X' + (f' (мм: 0–{x_size})' if x_size else ' (ячейки)'))
    plt.ylabel('Y' + (f' (мм: 0–{y_size})' if y_size else ' (ячейки)'))
    
    # Добавляем сетку
    ax.grid(True, linestyle=':', alpha=0.8)
    
    # Статистика на графике
    stats_text = (f"Среднее: {np.mean(data_2d):.3f}\n"
                  f"Мин: {np.min(data_2d):.3f}\n"
                  f"Макс: {np.max(data_2d):.3f}\n"
                  f"⚠ >порога: {np.sum(data_2d >= threshold)}/{x*y} "
                  f"({100*np.sum(data_2d >= threshold)/(x*y):.1f}%)")
    
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.02, 1.5, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"💾 Сохранено: {output_path}")
    else:
        plt.show()
    
    plt.close()
    return fig


def process_directory(input_dir, output_dir='./output', threshold=1.0):
    """Обработка всех .raw файлов в директории"""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    raw_files = list(Path(input_dir).glob('thick_*.raw'))
    print(f"🔍 Найдено файлов: {len(raw_files)}")
    
    results = []
    
    for filepath in raw_files:
        print(f"\n📁 Обработка: {filepath.name}")
        try:
            # Чтение данных
            raw_data = read_raw_file(str(filepath))
            data = raw_data['data']
            threshold = raw_data['thicknom']*0.9
            # Визуализация
            output_path = os.path.join(output_dir, f'{filepath.stem}.png')
            visualize_heatmap(data, output_path, threshold=threshold,
                            title=f"ID: {filepath.stem.split('_')[-1]}")
            
            # Сбор статистики
            stats = {
                'file': filepath.name,
                'shape': data.shape,
                'mean': float(np.mean(data)),
                'std': float(np.std(data)),
                'min': float(np.min(data)),
                'max': float(np.max(data)),
                'defect_pct': float(100 * np.sum(data >= threshold) / data.size)
            }
            results.append(stats)
            print(f"✅ {filepath.name}: ⚠ дефекты {stats['defect_pct']:.1f}%")
            
        except Exception as e:
            print(f"❌ Ошибка обработки {filepath.name}: {e}")
    
    # Сводная статистика
    if results:
        print("\n📊 Сводная статистика:")
        print(f"{'Файл':<25} {'Среднее':>8} {'Стд.откл.':>10} {'Дефекты':>8}")
        print("-" * 55)
        for r in results:
            print(f"{r['file']:<25} {r['mean']:8.3f} {r['std']:10.3f} {r['defect_pct']:7.1f}%")
    
    return results

def process(test_file):
    INPUT_DIR = './thicks_export_2'      # Директория с .raw файлами
    OUTPUT_DIR = './visualization_results_2'  # Директория для результатов
    THRESHOLD = 5.0                     # Порог дефекта (как в Java-коде)
    
    # Обработка одного файла (для теста)
    #test_file = './thicks_export/thick_409167.raw'
    if os.path.exists(test_file):
        print(f"🧪 Тестовый режим: {test_file}")
        raw_data = read_raw_file(test_file)
        data = raw_data['data']
        THRESHOLD = raw_data['thicknom']*0.9
        print(f"📐 Размер матрицы: {data.shape}")
        print(f"📈 Диапазон значений: [{np.min(data):.3f}, {np.max(data):.3f}]")
        print(f"📊 Среднее: {np.mean(data):.3f} ± {np.std(data):.3f}")
        print(f"⚠ Ячеек выше порога ({THRESHOLD}): "
              f"{np.sum(data >= THRESHOLD)}/{data.size} "
              f"({100*np.sum(data >= THRESHOLD)/data.size:.1f}%)")
        
        # Визуализация
        visualize_heatmap(data, output_path='./test_output.png', 
                         threshold=THRESHOLD, title='Тестовый образец')
    # Массовая обработка (раскомментируйте для использования)
    process_directory(INPUT_DIR, OUTPUT_DIR, threshold=THRESHOLD)

# 🔧 Пример использования
if __name__ == '__main__':
    # Настройки
    process('./thicks_export/thick_409167.raw')
    
    # Массовая обработка (раскомментируйте для использования)
    # process_directory(INPUT_DIR, OUTPUT_DIR, threshold=THRESHOLD)

🧪 Тестовый режим: ./thicks_export/thick_409167.raw
📊 Заголовок: 203×8
📐 Размер матрицы: (203, 8)
📈 Диапазон значений: [0.000, 5.650]
📊 Среднее: 5.243 ± 1.374
⚠ Ячеек выше порога (4.95): 1520/1624 (93.6%)


C:\Users\strel\AppData\Local\Temp\ipykernel_18092\1596001721.py:122: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\strel\AppData\Local\Temp\ipykernel_18092\1596001721.py:122: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\strel\AppData\Local\Temp\ipykernel_18092\1596001721.py:125: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=150, bbox_inches='tight')
C:\Users\strel\AppData\Local\Temp\ipykernel_18092\1596001721.py:125: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=150, bbox_inches='tight')


💾 Сохранено: ./test_output.png
🔍 Найдено файлов: 3510

📁 Обработка: thick_378463.raw
📊 Заголовок: 202×8
💾 Сохранено: ./visualization_results_2\thick_378463.png
✅ thick_378463.raw: ⚠ дефекты 92.8%

📁 Обработка: thick_378464.raw
📊 Заголовок: 202×8
💾 Сохранено: ./visualization_results_2\thick_378464.png
✅ thick_378464.raw: ⚠ дефекты 93.0%

📁 Обработка: thick_378465.raw
📊 Заголовок: 203×8
💾 Сохранено: ./visualization_results_2\thick_378465.png
✅ thick_378465.raw: ⚠ дефекты 93.5%

📁 Обработка: thick_378466.raw
📊 Заголовок: 203×8
💾 Сохранено: ./visualization_results_2\thick_378466.png
✅ thick_378466.raw: ⚠ дефекты 96.0%

📁 Обработка: thick_378467.raw
📊 Заголовок: 204×8
💾 Сохранено: ./visualization_results_2\thick_378467.png
✅ thick_378467.raw: ⚠ дефекты 93.6%

📁 Обработка: thick_378468.raw
📊 Заголовок: 205×8
💾 Сохранено: ./visualization_results_2\thick_378468.png
✅ thick_378468.raw: ⚠ дефекты 93.6%

📁 Обработка: thick_378469.raw
📊 Заголовок: 203×8
💾 Сохранено: ./visualization_results_2\thick